# Experimento B: más épocas + menos peso a las clases raras

En la referencia, las clases raras (`¿`, `,`, `?`, MiXta, MAYÚS) tienen recall mucho mayor que precisión: el peso de clase `(N / (K · n_clase)) ** class_weight_power` hace que se predigan de más. Cambios respecto de la referencia: **10 épocas** (como en A) y **`class_weight_power` = 0.25 en lugar de 0.5**. Comparar contra A para aislar el efecto del peso de clase.

Referencia: `models/2026-11-16__21-24-10` (5 épocas, `class_weight_power=0.5`). Los dos modelos subajustan: la loss de train sigue bajando y su mejor época es la 4 o la 5.

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**. Los resultados se guardan en Drive (`MyDrive/tp2-models`).

## 1. Setup

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "main"
REPO = "/content/TP2-Aprendizaje-Automatico-UBA"

if IN_COLAB:
    # Los resultados van a Drive: la máquina de Colab se borra al cerrar la sesión
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(REPO).exists():
        !git clone https://github.com/SebasAndres/TP2-Aprendizaje-Automatico-UBA.git {REPO}
    %cd {REPO}
    !git fetch origin && git checkout {BRANCH} && git pull
    !pip install -q uv && uv sync
    !mkdir -p /content/drive/MyDrive/tp2-models
    !if [ ! -L models ]; then rm -rf models && ln -s /content/drive/MyDrive/tp2-models models; fi
else:
    # Local: el notebook corre desde notebooks/experiments/, subimos hasta la raíz del repo
    %cd ../..

!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## 2. Configuración

In [ ]:
EPOCHS = 10
SIZE = 1_000_000  # instancias por fuente: igual que la referencia (en la práctica, todo el dataset)
CLASS_WEIGHT_POWER = 0.25  # referencia: 0.5 (src/constants.py)

# Pisamos la constante antes de importar el pipeline, sin tocar el código del repo.
# run_config la lee de la misma constante, así que queda registrada en metrics.json.
PIPELINE = (
    "MPLBACKEND=Agg uv run python -c "
    f"'import runpy, sys; import src.constants as c; c.CLASS_WEIGHT_POWER = {CLASS_WEIGHT_POWER}; "
    "sys.argv = [\"src.pipeline\"] + sys.argv[1:]; runpy.run_module(\"src.pipeline\", run_name=\"__main__\")'"
)

# F1 macro en CONTROL de la corrida de referencia (models/2026-11-16__21-24-10: 5 épocas, class_weight_power=0.5)
BASELINE_CONTROL = {
    ("CapitalizationProblem", "bidirectional_rnn", "capitalizacion"): 0.605,
    ("CapitalizationProblem", "joint_conv_bilstm", "capitalizacion"): 0.614,
    ("PunctuationProblem", "bidirectional_rnn", "punt_inicial"): 0.789,
    ("PunctuationProblem", "bidirectional_rnn", "punt_final"): 0.546,
    ("PunctuationProblem", "joint_conv_bilstm", "punt_inicial"): 0.713,
    ("PunctuationProblem", "joint_conv_bilstm", "punt_final"): 0.594,
}
RUNS: dict[str, Path | None] = {}

## 3. Entrenamiento

Una celda por modelo: si Colab corta la sesión, lo que ya terminó queda guardado en Drive (`MyDrive/tp2-models/<fecha>__<hora>/`).

### `bidirectional_rnn`

In [ ]:
before = set(Path("models").iterdir())
!{PIPELINE} --experiments bidirectional_rnn --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}
new_runs = sorted(set(Path("models").iterdir()) - before)
RUNS["bidirectional_rnn"] = new_runs[-1] if new_runs else None
print("Resultados en:", RUNS["bidirectional_rnn"])

### `joint_conv_bilstm`

In [ ]:
before = set(Path("models").iterdir())
!{PIPELINE} --experiments joint_conv_bilstm --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}
new_runs = sorted(set(Path("models").iterdir()) - before)
RUNS["joint_conv_bilstm"] = new_runs[-1] if new_runs else None
print("Resultados en:", RUNS["joint_conv_bilstm"])

## 4. Resultados

Se compara sobre **control** (test ya se usó para elegir la época). Si se reinició el runtime, completar `RUNS` a mano con las carpetas de Drive, ej: `RUNS = {"bidirectional_rnn": Path("models/2026-11-18__10-00-00")}`.

In [ ]:
import json


def task_rows(run_dir: Path) -> list[tuple[str, str, str, float, float]]:
    """(problema, modelo, tarea, F1 test, F1 control) de cada tarea de una corrida."""
    report = json.loads((run_dir / "metrics.json").read_text())
    rows = []
    for problem, models in report["problems"].items():
        for model, r in models.items():
            for task, v in r["test"].items():
                control = r["control"][task]["f1_macro"] if r.get("control") else float("nan")
                rows.append((problem, model, task, v["f1_macro"], control))
    return rows


print(f"{'problema':22} {'modelo':18} {'tarea':15} {'test':>6} {'control':>8} {'baseline':>9} {'Δ control':>10}")
for experiment, run_dir in RUNS.items():
    if run_dir is None or not (run_dir / "metrics.json").exists():
        print(f"{experiment}: sin metrics.json (¿se cortó la corrida?)")
        continue
    for problem, model, task, test, control in task_rows(run_dir):
        base = BASELINE_CONTROL.get((problem, model, task), float("nan"))
        print(f"{problem:22} {model:18} {task:15} {test:6.3f} {control:8.3f} {base:9.3f} {control - base:+10.3f}")